
# Energy Calculation from Measurement Sweeps

This notebook reproduces the energy computation workflow used in the automated test bench. It loads voltage and current traces gathered during a phase-shift sweep, aligns them with the configuration metadata, and derives switching-off energy metrics alongside diagnostic plots.



## Method Summary

We reuse the following relationships for each phase step:

- Weighted average current based on instantaneous power samples:
  
  $$I_{\text{avg}} = \frac{\sum_{k=1}^{N} I_k V_k}{\sum_{k=1}^{N} V_k}$$

- Average voltage during the same interval:
  
  $$V_{\text{avg}} = \frac{1}{N} \sum_{k=1}^{N} V_k$$

- Experimental power:
  
  $$P = I_{\text{avg}} \times V_{\text{avg}}$$

- Theoretical current limit derived from inductive energy storage (with the phase angle converted to radians):
  
  $$I_{\text{lim}} = \frac{\phi}{180} \cdot \frac{V_{\text{level}}}{4 f L_{\text{mH}}}$$

- Switching-off energy per cycle:
  
  $$E_{\text{off}} = \frac{1}{4} \cdot \frac{P - I_{\text{lim}}^{2} R_{\text{tot}}}{f}$$

where:

- $I_k$, $V_k$ are instantaneous measurements;
- $f$ is the switching frequency;
- $L_{\text{mH}}$ and $R_{\text{tot}}$ are drawn from `base-config.json` (`InductorL` in mH, `TotalR` in Ohms);
- $\phi$ is the phase shift in degrees for the segment.



## Usage Instructions

1. Set the configuration paths and identifiers in the next code cell to point at your measurement folder (e.g., `PhaseShift/Voltage_40_50kHz`).
2. The notebook will auto-detect the matching voltage/current CSV files, the sweep JSON configuration, and the `results.json` file that stores peak current statistics.
3. Run the notebook sequentially to compute the per-step energy table and generate diagnostic plots.
4. Adjust the helper functions if your file naming deviates from the default export convention (`*-voltage.csv`, `*-current.csv`).


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-colorblind")


In [ ]:
# --- Configure paths for a specific sweep ---

# Root directory that contains the PhaseShift/DutyShift measurement folders
DATA_ROOT = Path("../PhaseShift").resolve()

# Base configuration file containing TotalR and InductorL
BASE_CONFIG_PATH = Path("../base-config.json").resolve()

# Voltage level (e.g., 40 for 40 V) used in the sweep
VOLTAGE_LEVEL = 40

# Switching frequency in Hz for the sweep (e.g., 50_000 for 50 kHz)
FREQUENCY_HZ = 50_000

# Name of the phase-shift configuration JSON file stored in the profile folder
PHASE_PROFILE_NAME = f"PhaseShift_Voltage_{VOLTAGE_LEVEL}V_Freq{FREQUENCY_HZ // 1000}kHz.json"

# Name of the results summary JSON file saved by the supervisor
RESULTS_FILENAME = "results.json"

PROFILE_ROOT = DATA_ROOT / f"Voltage_{VOLTAGE_LEVEL}_{FREQUENCY_HZ // 1000}kHz"
PHASE_PROFILE_PATH = PROFILE_ROOT / PHASE_PROFILE_NAME
RESULTS_PATH = PROFILE_ROOT / RESULTS_FILENAME


In [ ]:
# --- Helpers to load measurement series in multiple possible formats ---

def load_series_from_file(path: Path, column_name: str) -> pd.Series:
    """Load a voltage or current series from the export file.

    Handles JSON-encoded lists in a single cell, plain single-column CSV, or CSV arrays.
    """
    if not path.exists():
        raise FileNotFoundError(f"Measurement file not found: {path}")

    raw = path.read_text().strip()
    if raw.startswith("["):
        data = json.loads(raw)
        return pd.Series(data, name=column_name)

    df = pd.read_csv(path, header=None)
    if df.shape[0] == 1 and df.shape[1] >= 1:
        # JSON stored in the first cell
        data = json.loads(df.iloc[0, 0])
        return pd.Series(data, name=column_name)
    if df.shape[1] == 1:
        return pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna().reset_index(drop=True).rename(column_name)

    raise ValueError(f"Unrecognized file format for {path}")


def autodetect_measurement_files(profile_root: Path) -> Tuple[Path, Path]:
    """Return the first (voltage, current) CSV pair found in the profile folder."""
    voltage_candidates = sorted(profile_root.glob("*-voltage*.csv"))
    current_candidates = sorted(profile_root.glob("*-current*.csv"))
    if not voltage_candidates or not current_candidates:
        raise FileNotFoundError(
            "Could not find measurement CSV files. Check naming or update autodetect_measurement_files()."
        )
    return voltage_candidates[0], current_candidates[0]


VOLTAGE_FILE, CURRENT_FILE = autodetect_measurement_files(PROFILE_ROOT)
VOLTAGE_FILE, CURRENT_FILE


In [ ]:
# --- Load configuration and measurement data ---

base_cfg = json.loads(BASE_CONFIG_PATH.read_text())
TotalR = base_cfg.get("TotalR")
InductorL_mH = base_cfg.get("InductorL")
if TotalR is None or InductorL_mH is None:
    raise KeyError("TotalR or InductorL is missing from base-config.json. Update the file before proceeding.")

with open(PHASE_PROFILE_PATH) as fp:
    phase_profile = json.load(fp)

with open(RESULTS_PATH) as fp:
    result_summary = json.load(fp)

voltage_series = load_series_from_file(VOLTAGE_FILE, "Voltage (V)")
current_series = load_series_from_file(CURRENT_FILE, "Current (A)")

measurement_df = pd.DataFrame({
    "Voltage (V)": voltage_series,
    "Current (A)": current_series
}).dropna()

measurement_df.head()


In [ ]:
# --- Derive per-phase statistics ---

nb_points = phase_profile["nbPointsI"]
phase_tab = phase_profile.get("PhaseTab")
if phase_tab is None:
    phase_tab = list(range(phase_profile["PhaseInit"], phase_profile["PhaseFinal"] + 1, phase_profile["PhaseStep"]))

if len(phase_tab) < nb_points:
    print(f"[Warning] PhaseTab shorter than nbPointsI. Adjusting nb_points from {nb_points} to {len(phase_tab)}.")
    nb_points = len(phase_tab)
else:
    phase_tab = phase_tab[:nb_points]

samples_per_step = (len(measurement_df) // nb_points)
if samples_per_step == 0:
    raise ValueError("Insufficient samples to split by phase step. Verify the CSV contents and nbPointsI.")

trimmed_len = samples_per_step * nb_points
measurement_df = measurement_df.iloc[:trimmed_len].reset_index(drop=True)

phase_array = np.repeat(phase_tab, samples_per_step)
measurement_df["Phase (°)"] = phase_array

phase_groups = measurement_df.groupby("Phase (°)")

rows = []
for idx, (phase_deg, group) in enumerate(phase_groups, start=1):
    voltage = group["Voltage (V)"].to_numpy()
    current = group["Current (A)"].to_numpy()
    power_samples = voltage * current

    denom = voltage.sum()
    i_avg = power_samples.sum() / denom if denom != 0 else np.nan
    v_avg = voltage.mean()
    p_avg = i_avg * v_avg

    i_lim = (phase_deg / 180.0) * VOLTAGE_LEVEL / (4.0 * FREQUENCY_HZ * (InductorL_mH * 1e-3))
    e_off = 0.25 * (p_avg - (i_lim ** 2) * TotalR) / FREQUENCY_HZ

    peak_key = f"Data{idx}"
    i_peak = (
        result_summary.get(peak_key, {})
        .get("2", {})
        .get("value", np.nan)
    )

    rows.append({
        "Step": idx,
        "Phase (°)": phase_deg,
        "I_avg (A)": i_avg,
        "V_avg (V)": v_avg,
        "P_avg (W)": p_avg,
        "I_lim (A)": i_lim,
        "E_off (J)": e_off,
        "I_peak (A)": i_peak,
    })

energy_df = pd.DataFrame(rows)
energy_df


In [ ]:
# --- Plot energy vs peak current and instantaneous power timeline ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(energy_df["I_peak (A)"], energy_df["E_off (J)"], marker="o")
axes[0].set_title("E_off vs I_peak")
axes[0].set_xlabel("I_peak (A)")
axes[0].set_ylabel("E_off (J)")
axes[0].grid(True)

axes[1].plot((measurement_df.index + 1), measurement_df["Voltage (V)"] * measurement_df["Current (A)"])
axes[1].set_title("Instantaneous Power Samples")
axes[1].set_xlabel("Sample Index")
axes[1].set_ylabel("Power (W)")
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# --- Persist the computed table for downstream analysis ---

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_filename = OUTPUT_DIR / f"energy_summary_{VOLTAGE_LEVEL}V_{FREQUENCY_HZ // 1000}kHz.csv"
energy_df.to_csv(output_filename, index=False)

output_filename
